In [1]:
cd ..

/home/serafeim/Desktop/bet


In [2]:

from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity
from prepare_data_inference import prepare_data_inference


In [3]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 20:49:08 WARN Utils: Your hostname, serafeim-virtual-machine, resolves to a loopback address: 127.0.1.1; using 10.100.2.34 instead (on interface ens192)
26/03/03 20:49:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 20:49:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')


In [5]:
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')

In [6]:
silver_money_events.filter(F.col('event_ts')<='2024-06-20').filter(F.col('player_idx')==233).orderBy(F.col('event_ts').desc()).show()

+--------------------+-------------------+----------+-------------+------------------+----------+
|            event_id|           event_ts|event_type|signed_amount| balance_after_txn|player_idx|
+--------------------+-------------------+----------+-------------+------------------+----------+
|ab302da2-9f1a-425...|2024-06-19 23:47:29|   session|         90.7| 6098.799999999999|       233|
|3064f2d6-a4e7-41d...|2024-06-17 04:25:43|   session|        94.82| 6008.099999999999|       233|
|770b64e5-fbf6-4ce...|2024-06-17 02:38:46|   session|         4.88|           5913.28|       233|
|23f84075-3e1f-425...|2024-06-16 09:25:42|   session|       108.95|            5908.4|       233|
|5c05010c-6494-497...|2024-06-15 11:39:02|   session|        17.58|           5799.45|       233|
|aa67a424-2872-432...|2024-06-10 21:37:13|   session|       100.51|           5781.87|       233|
|e6695380-3be8-431...|2024-06-08 03:48:46|   session|        90.52|           5681.36|       233|
|aff098c9-df3b-483..

In [26]:
test_date = '2024-05-20'
data_inference = prepare_data_inference(test_date)

failed_withdrawals_30d
deposit_count_30d
withdrawal_count_30d
withdrawal_ratio


In [27]:
def compare(df1,df2): 
   return (( df1.exceptAll(df2).count() == 0) & (df2.exceptAll(df1).count() == 0))

m1 = player_behavior.filter(F.col('reference_date')==test_date).select('player_idx').join(data_inference, how='inner', on='player_idx', )
compare(m1, data_inference)

True

In [28]:
from pyspark.sql.types import NumericType

ex_ids = player_behavior.filter(F.col('reference_date')==test_date).select('player_idx').join(data_inference.select('player_idx'), on='player_idx', how='left_anti')
ex_frame = player_behavior.filter(F.col('reference_date')==test_date).join(ex_ids, on='player_idx', how='inner')

# Identify numeric columns
numeric_cols = [
    field.name
    for field in ex_frame.schema.fields
    if isinstance(field.dataType, NumericType)
]

# Compute sums
df_sum = ex_frame.select([
    F.sum(F.col(c)).alias(c)
    for c in numeric_cols
])

df_sum.show()

+----------+--------------+---------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|player_idx|balance_7d_ago|balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|
+----------+--------------+---------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+
|     16693|      53948.56|       39974.43|  -86.99000000000001|              14429.1|              0|               0.0|             228|        53947